# Taller 5 — Física Matemática I
## Visualización Computacional de los Problemas

**Universidad de Antioquia**  
**Autor:** Daniel Soto Villada — Estudiante de Astronomía  
**Curso:** Física Matemática I

---

Este notebook complementa la solución analítica del Taller 5 con visualizaciones en Python. Se cubren los seis problemas del taller:

1. **Problema 1:** Valor esperado real $\langle A \rangle \in \mathbb{R}$ → operadores hermitianos  
2. **Problema 2:** Autoadjunción de $\hat{p}$ y $\hat{L}$ y condiciones de hermiticidad  
3. **Problema 3:** Factores integrantes y forma autoadjunta de EDOs  
4. **Problema 4:** Autofunciones y autovalores de problemas de Sturm-Liouville  
5. **Problema 5:** Ecuación de Euler — no ortogonalidad  
6. **Problema 6:** Funciones especiales en forma autoadjunta  

---


## Importación de Librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm
from scipy.special import (jv, lpmv, eval_hermite, eval_laguerre,
                            eval_genlaguerre, gegenbauer,
                            sph_harm_y, jn_zeros)
from scipy.linalg import eigh
import warnings
warnings.filterwarnings('ignore')

# Estilo global
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f8f8',
    'axes.grid': True, 'grid.alpha': 0.4, 'lines.linewidth': 2,
    'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 13,
    'legend.fontsize': 11, 'xtick.labelsize': 11, 'ytick.labelsize': 11,
})
COLORS = plt.cm.tab10.colors
print("Librerias importadas correctamente")


---
## Problema 1: Valor Esperado Real e Índole Hermitiana del Operador

**Enunciado:** Muestre que $\langle A \rangle \in \mathbb{R}$ para cualquier estado $|\psi\rangle$ implica $A = A^\dagger$.

### Teoría

$$\langle A \rangle = \langle \psi | A | \psi \rangle = \int \psi^*(x)\, A\,\psi(x)\, dx$$

Imponiendo $\langle A \rangle = \langle A \rangle^*$ para *cualquier* $\psi$ y usando superposiciones lineales $\Psi = \phi + \lambda\psi$, se demuestra que $A = A^\dagger$.

Las visualizaciones muestran:
1. Autovalores reales (operador Hermítico) vs. complejos (no Hermítico)  
2. Valor esperado $\langle\hat{x}\rangle(t)$ real para un oscilador armónico cuántico  
3. Comparación con operador anti-Hermítico cuyo valor esperado es imaginario  


In [ ]:
# P1-1: Autovalores reales vs complejos
np.random.seed(42)

# Matriz hermítica 4x4 (real simétrica)
H_re = np.random.randn(4, 4)
A_herm  = H_re + H_re.T

# Matriz no hermítica
A_nherm = np.random.randn(4, 4) + 1j * np.random.randn(4, 4)

eigs_h  = np.linalg.eigvals(A_herm)
eigs_nh = np.linalg.eigvals(A_nherm)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Problema 1: Autovalores de Matrices Hermiticas vs. No Hermiticas',
             fontsize=15, fontweight='bold')

ax = axes[0]
ax.scatter(eigs_h.real, eigs_h.imag, s=200, c='steelblue', zorder=5,
           edgecolors='k', linewidths=0.8, label='Autovalores')
ax.axhline(0, color='black', lw=1.0, ls='--', alpha=0.7)
ax.axvline(0, color='black', lw=1.0, ls='--', alpha=0.7)
ax.set_title(r'Matriz Hermitica $A = A^\dagger$', color='steelblue')
ax.set_xlabel(r'Re($\lambda$)'); ax.set_ylabel(r'Im($\lambda$)')
for ev in eigs_h:
    ax.annotate(f'  ({ev.real:.2f}, {ev.imag:.2f}i)',
                xy=(ev.real, ev.imag), fontsize=9, color='navy')
ax.legend()

ax = axes[1]
ax.scatter(eigs_nh.real, eigs_nh.imag, s=200, c='tomato', zorder=5,
           edgecolors='k', linewidths=0.8, label='Autovalores')
ax.axhline(0, color='black', lw=1.0, ls='--', alpha=0.7)
ax.axvline(0, color='black', lw=1.0, ls='--', alpha=0.7)
ax.set_title(r'Matriz No Hermitica ($A \neq A^\dagger$)', color='tomato')
ax.set_xlabel(r'Re($\lambda$)'); ax.set_ylabel(r'Im($\lambda$)')
for ev in eigs_nh:
    ax.annotate(f'  ({ev.real:.2f}, {ev.imag:.2f}i)',
                xy=(ev.real, ev.imag), fontsize=9, color='darkred')
ax.legend()

plt.tight_layout()
plt.show()
print("Autovalores Hermitica (todos reales):", np.round(eigs_h.real, 4))
print("Partes imaginarias (aprox 0):", np.round(eigs_h.imag, 10))


In [ ]:
# P1-2: Valor esperado <x>(t) para oscilador armonico cuantico
N = 5
H_ho = np.diag([n + 0.5 for n in range(N)])
a    = np.diag(np.sqrt(np.arange(1, N)), k=1)
x_op = (a + a.T) / np.sqrt(2)

psi0 = np.array([1, 1, 1, 0, 0], dtype=complex) / np.sqrt(3)
times = np.linspace(0, 6*np.pi, 500)

exp_x = []
for t in times:
    U     = np.diag(np.exp(-1j * H_ho.diagonal() * t))
    psi_t = U @ psi0
    exp_x.append((psi_t.conj() @ (x_op @ psi_t)).real)
exp_x = np.array(exp_x)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(times / np.pi, exp_x, color='steelblue', lw=2)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_title(r'Problema 1: $\langle\hat{x}\rangle(t)$ — operador Hermitco → valor real',
             fontweight='bold')
ax.set_xlabel(r'Tiempo $t/\pi\;[1/\omega]$')
ax.set_ylabel(r'$\langle\hat{x}\rangle$ [adim.]')
ax.fill_between(times/np.pi, exp_x, alpha=0.15, color='steelblue')
plt.tight_layout()
plt.show()
print(f"Max |Im(<x>)| = {np.max(np.abs(np.imag(exp_x))):.2e}  (aprox 0 — confirma hermiticidad)")


In [ ]:
# P1-3: Operador anti-Hermitco — valor esperado imaginario
B = x_op * 1j    # B = i*x_hat, anti-Hermitco: B† = -B

exp_B = []
for t in times:
    U = np.diag(np.exp(-1j * H_ho.diagonal() * t))
    psi_t = U @ psi0
    exp_B.append(psi_t.conj() @ (B @ psi_t))
exp_B = np.array(exp_B)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle('Problema 1: Operador Anti-Hermitco $B = i\\hat{x}$\n'
             'El valor esperado tiene parte imaginaria no nula', fontweight='bold')

axes[0].plot(times/np.pi, exp_B.real, color='steelblue',
             label=r'Re$\langle B \rangle$')
axes[0].set_ylabel('Parte Real'); axes[0].legend()

axes[1].plot(times/np.pi, exp_B.imag, color='tomato',
             label=r'Im$\langle B \rangle$')
axes[1].set_ylabel('Parte Imaginaria'); axes[1].legend()
axes[1].set_xlabel(r'Tiempo $t/\pi$')

for ax in axes:
    ax.axhline(0, color='gray', lw=0.8, ls='--')

plt.tight_layout()
plt.show()


---
## Problema 2: Operadores $\hat{p}$ y $\hat{L}$ son Autoadjuntos

**Enunciado:** Mostrar que $\vec{p}=-i\hbar\nabla$ y $\vec{L}=\vec{r}\times\vec{p}$ son autoadjuntos.
¿Qué condiciones deben satisfacer para ser hermíticos?

### Idea clave

Al integrar por partes aparece un **término de frontera**:

$$\langle \psi | p_x \phi \rangle = \langle p_x \psi | \phi \rangle - i\hbar\left[\psi^*(x)\phi(x)\right]_{-\infty}^{+\infty}$$

Para hermiticidad: ese término debe anularse ($\psi \to 0$ en el infinito).  
Para $L_z = -i\hbar\partial/\partial\phi$: periodicidad $\psi(\phi+2\pi)=\psi(\phi)$.


In [ ]:
# P2-1: Caida de funciones de onda y verificacion de hermiticidad de p_x
x = np.linspace(-6, 6, 1000)

def coherent(x, x0=2.0, p0=1.5, sigma=1.0):
    return (np.exp(-(x-x0)**2/(2*sigma**2)) * np.exp(1j*p0*x)
            / (np.pi*sigma**2)**0.25)

psi = coherent(x); phi = coherent(x, x0=-1.5, p0=-0.8, sigma=1.2)
hbar = 1.0; dx = x[1]-x[0]

px_psi = -1j*hbar*np.gradient(psi, dx)
px_phi = -1j*hbar*np.gradient(phi, dx)

lhs = np.trapezoid(psi.conj() * px_phi, x)
rhs = np.trapezoid(px_psi.conj() * phi, x)
boundary = psi.conj() * phi

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(r'Problema 2: Operador Momento $\hat{p}_x = -i\hbar\partial_x$'
             '\nVerificacion de Hermiticidad', fontweight='bold')

axes[0,0].plot(x, np.abs(psi)**2, 'steelblue', label=r'$|\psi|^2$')
axes[0,0].plot(x, np.abs(phi)**2, 'tomato', label=r'$|\phi|^2$')
axes[0,0].set_title('Densidades de probabilidad'); axes[0,0].legend()
axes[0,0].set_xlabel('$x$'); axes[0,0].set_ylabel(r'$|\psi(x)|^2$')

axes[0,1].plot(x, psi.real, 'steelblue', label=r'Re $\psi$')
axes[0,1].plot(x, psi.imag, 'steelblue', ls='--', label=r'Im $\psi$')
axes[0,1].plot(x, phi.real, 'tomato', label=r'Re $\phi$')
axes[0,1].plot(x, phi.imag, 'tomato', ls='--', label=r'Im $\phi$')
axes[0,1].set_title('Funciones de onda'); axes[0,1].legend(fontsize=9)
axes[0,1].set_xlabel('$x$'); axes[0,1].set_ylabel('Amplitud')

axes[1,0].plot(x, (psi.conj()*px_phi).real, 'steelblue',
               label=r'Re$[\psi^*(p_x\phi)]$')
axes[1,0].plot(x, (px_psi.conj()*phi).real, 'tomato', ls='--',
               label=r'Re$[(p_x\psi)^*\phi]$')
axes[1,0].set_title('Integrandos del producto interno'); axes[1,0].legend()
axes[1,0].set_xlabel('$x$'); axes[1,0].set_ylabel('Amplitud')

axes[1,1].plot(x, np.abs(boundary), 'purple', label=r'$|\psi^*\phi|$')
axes[1,1].set_title(r'Termino de frontera $|\psi^*(x)\phi(x)|$')
axes[1,1].set_xlabel('$x$'); axes[1,1].set_ylabel('Modulo'); axes[1,1].legend()
bterm = abs(boundary[-1] - boundary[0])
axes[1,1].text(0, axes[1,1].get_ylim()[1]*0.85,
               f'[psi* phi] en extremos = {bterm:.2e}  aprox 0',
               ha='center', fontsize=11, bbox=dict(facecolor='lightyellow'))

plt.tight_layout(); plt.show()
print(f"<psi|p_x phi> = {lhs:.6f}")
print(f"<p_x psi|phi> = {rhs:.6f}")
print(f"Diferencia (aprox 0): {abs(lhs-rhs):.2e}")


In [ ]:
# P2-2: Periodicidad en phi para L_z
phi_angle = np.linspace(0, 4*np.pi, 1000)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(r'Problema 2: $L_z = -i\hbar\,\partial/\partial\phi$'
             '\nCondicion de hermiticidad: periodicidad', fontweight='bold')

for m, color in zip([0,1,2,3], COLORS):
    psi_m = np.exp(1j*m*phi_angle)
    axes[0,0].plot(phi_angle/np.pi, psi_m.real, color=color, label=f'm={m}')
axes[0,0].set_title(r'Re$[e^{im\phi}]$ — m entero (univaluada)')
axes[0,0].set_xlabel(r'$\phi/\pi$'); axes[0,0].set_ylabel('Amplitud')
axes[0,0].axvline(2, color='k', ls=':', alpha=0.7); axes[0,0].legend()

m_bad = 0.7
psi_bad = np.exp(1j*m_bad*phi_angle)
axes[0,1].plot(phi_angle/np.pi, psi_bad.real, 'tomato',
               label=f'm={m_bad} (no entero)')
axes[0,1].axvline(2, color='k', ls=':', alpha=0.7, label=r'$2\pi$')
axes[0,1].set_title(r'Re$[e^{im\phi}]$ — m NO entero (NO univaluada)', color='tomato')
axes[0,1].set_xlabel(r'$\phi/\pi$'); axes[0,1].set_ylabel('Amplitud')
axes[0,1].legend()

ms = np.arange(-4,5)
axes[1,0].stem(ms, ms, basefmt='gray', linefmt='steelblue', markerfmt='bo')
axes[1,0].set_title(r'Autovalores de $L_z$: $m\hbar$, $m\in\mathbb{Z}$')
axes[1,0].set_xlabel('$m$'); axes[1,0].set_ylabel(r'$L_z/\hbar$')

phi_2pi = np.linspace(0, 2*np.pi, 500)
for l, m_v, color in [(1,0,'steelblue'),(1,1,'tomato'),(2,0,'green'),(2,2,'purple')]:
    Y = sph_harm_y(l, m_v, np.pi/2, phi_2pi)
    axes[1,1].plot(phi_2pi/np.pi, np.abs(Y)**2, color=color,
                   label=f'$|Y_{l}^{{{m_v}}}|^2$')
axes[1,1].set_title(r'$|Y_l^m(\theta=\pi/2,\phi)|^2$ — periodicidad garantizada')
axes[1,1].set_xlabel(r'$\phi/\pi$'); axes[1,1].set_ylabel('Densidad angular')
axes[1,1].legend()

plt.tight_layout(); plt.show()


In [ ]:
# P2-3: Armonicos esfericos |Y_lm|^2 en 3D
theta = np.linspace(0, np.pi, 100)
phi_a = np.linspace(0, 2*np.pi, 100)
THETA, PHI = np.meshgrid(theta, phi_a)

fig = plt.figure(figsize=(16, 10))
fig.suptitle(r'Problema 2: $|Y_l^m(\theta,\phi)|^2$ — Armonicos Esfericos'
             '\n(autofunciones de $L^2$ y $L_z$)',
             fontsize=14, fontweight='bold')

cases = [(0,0),(1,0),(1,1),(2,0),(2,1),(2,2)]
for idx, (l, m_v) in enumerate(cases):
    ax = fig.add_subplot(2, 3, idx+1, projection='3d')
    Y  = sph_harm_y(l, m_v, THETA, PHI)
    R  = np.abs(Y)**2
    X  = R * np.sin(THETA) * np.cos(PHI)
    Y3 = R * np.sin(THETA) * np.sin(PHI)
    Z  = R * np.cos(THETA)
    ax.plot_surface(X, Y3, Z, facecolors=cm.viridis(R/(R.max()+1e-10)),
                    alpha=0.9, linewidth=0)
    ax.set_title(f'$l={l},\\ m={m_v}$', fontsize=11)
    ax.set_axis_off()

plt.tight_layout(); plt.show()


---
## Problema 3: Ecuaciones en Forma Autoadjunta

**Enunciado:** Expresar las siguientes ecuaciones en forma autoadjunta $\frac{d}{dx}[q(x)y'] + \lambda p(x) y = 0$:

| Caso | $q(x)$ | $p(x)$ (factor de peso) |
|------|--------|------------------------|
| (a)  | $(1-x^2)^{-a}$ | $(1-x^2)^{-(a+1)}$ |
| (b)  | $(1-x^3)^{-1/3}$ | $(1-x^3)^{-4/3}$ |
| (c)  | $x^2 e^{-x^2}$ | $x e^{-x^2}$ |
| (d)  | $x^3 e^{-x^2}$ | $x^3 e^{-x^2}$ |
| (e)  | $x^2 e^{-x^2}$ | $e^{-x^2}$ |

El factor integrante es $p(x) = \frac{1}{a_2}\exp\!\left(\int\frac{a_1}{a_2}dx\right)$ y $q(x)=p(x)a_2(x)$.


In [ ]:
# P3: Factores q(x) y p(x) para cada caso
x_a = np.linspace(-0.95, 0.95, 500)
x_b = np.linspace(0.01, 0.99, 500)
x_pos = np.linspace(0.01, 3.0, 500)

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
fig.suptitle('Problema 3: Factores $q(x)$ y $p(x)$ en la Forma Autoadjunta',
             fontsize=15, fontweight='bold')

# Caso (a): distintos valores de a
ax = axes[0,0]
for a_val, ls in [(-1,'-'),(-0.5,'--'),(0,'-.'),(0.5,':'),(1,'-')]:
    q_a = np.clip((1-x_a**2)**(-a_val), -1, 15)
    ax.plot(x_a, q_a, ls=ls, label=f'$a={a_val}$')
ax.set_title(r'Caso (a): $q(x)=(1-x^2)^{-a}$')
ax.set_xlabel('$x$'); ax.set_ylabel('$q(x)$')
ax.set_ylim(-0.5, 8); ax.legend(fontsize=9)

ax = axes[0,1]
for a_val, ls in [(-1,'-'),(-0.5,'--'),(0,'-.'),(0.5,':'),(1,'-')]:
    p_a = np.clip((1-x_a**2)**(-a_val-1), -1, 15)
    ax.plot(x_a, p_a, ls=ls, label=f'$a={a_val}$')
ax.set_title(r'Caso (a): $p(x)=(1-x^2)^{-(a+1)}$')
ax.set_xlabel('$x$'); ax.set_ylabel('$p(x)$')
ax.set_ylim(-0.5, 8); ax.legend(fontsize=9)

# Caso (b)
ax = axes[1,0]
q_b = (1-x_b**3)**(-1/3); p_b = (1-x_b**3)**(-4/3)
ax.plot(x_b, np.clip(q_b,0,8), COLORS[2], label=r'$q(x)=(1-x^3)^{-1/3}$')
ax.plot(x_b, np.clip(p_b,0,8), COLORS[3], ls='--', label=r'$p(x)=(1-x^3)^{-4/3}$')
ax.set_title(r'Caso (b): dominio $x\in(0,1)$')
ax.set_xlabel('$x$'); ax.set_ylabel('Magnitud')
ax.set_ylim(0, 8); ax.legend()

# Casos c, d, e
q_c = x_pos**2*np.exp(-x_pos**2); p_c = x_pos*np.exp(-x_pos**2)
q_d = x_pos**3*np.exp(-x_pos**2); p_d = x_pos**3*np.exp(-x_pos**2)
q_e = x_pos**2*np.exp(-x_pos**2); p_e = np.exp(-x_pos**2)

ax = axes[1,1]
ax.plot(x_pos, q_c, COLORS[0], label=r'$q_c=x^2e^{-x^2}$')
ax.plot(x_pos, q_d, COLORS[1], label=r'$q_d=x^3e^{-x^2}$')
ax.plot(x_pos, q_e, COLORS[2], ls='--', label=r'$q_e=x^2e^{-x^2}$ (= $q_c$)')
ax.set_title('Casos (c), (d), (e): $q(x)$')
ax.set_xlabel('$x$'); ax.set_ylabel('$q(x)$'); ax.legend()

ax = axes[2,0]
ax.plot(x_pos, p_c, COLORS[0], label=r'$p_c=xe^{-x^2}$')
ax.plot(x_pos, p_d, COLORS[1], label=r'$p_d=x^3e^{-x^2}$')
ax.plot(x_pos, p_e, COLORS[2], ls='--', label=r'$p_e=e^{-x^2}$')
ax.set_title('Casos (c), (d), (e): $p(x)$ (factor de peso)')
ax.set_xlabel('$x$'); ax.set_ylabel('$p(x)$'); ax.legend()

# Comparacion factores de peso en [0,1]
ax = axes[2,1]
m = x_pos <= 1.0
ax.plot(x_a, np.clip((1-x_a**2)**(-1),0,4), COLORS[0], label='(a) $a=1$')
ax.plot(x_b, np.clip(p_b,0,4), COLORS[1], label='(b)')
ax.plot(x_pos[m], p_c[m], COLORS[2], label='(c)')
ax.plot(x_pos[m], p_d[m], COLORS[3], label='(d)')
ax.plot(x_pos[m], p_e[m], COLORS[4], label='(e)')
ax.set_title('Comparacion de factores de peso $p(x)$ en $[0,1]$')
ax.set_xlabel('$x$'); ax.set_ylabel('$p(x)$'); ax.set_ylim(-0.1,4); ax.legend(fontsize=9)

plt.tight_layout(); plt.show()


In [ ]:
# P3: Verificacion numerica de hermiticidad del operador de SL
# Caso (c): L[y] = d/dx[x^2 e^{-x^2} dy/dx], p(x) = x e^{-x^2}
x_v = np.linspace(0.05, 3.0, 2000)

f_test = x_v * np.exp(-x_v) * np.sin(np.pi*(x_v-0.05)/2.95)
g_test = x_v**2 * np.exp(-0.5*x_v**2) * np.sin(2*np.pi*(x_v-0.05)/2.95)

def apply_SL(y, xarr, q_arr):
    return np.gradient(q_arr * np.gradient(y, xarr), xarr)

p_c_v = x_v * np.exp(-x_v**2)
q_c_v = x_v**2 * np.exp(-x_v**2)
Lf = apply_SL(f_test, x_v, q_c_v)
Lg = apply_SL(g_test, x_v, q_c_v)

lhs_3 = np.trapezoid(f_test * Lg, x_v)
rhs_3 = np.trapezoid(Lf * g_test, x_v)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Problema 3: Verificacion Numerica de Hermiticidad\n'
             r'Caso (c): $L[y]=\frac{d}{dx}[x^2e^{-x^2}\frac{dy}{dx}]$',
             fontweight='bold')

axes[0].plot(x_v, f_test, COLORS[0], label='$f(x)$')
axes[0].plot(x_v, g_test, COLORS[1], label='$g(x)$')
axes[0].set_xlabel('$x$'); axes[0].set_ylabel('Amplitud')
axes[0].set_title('Funciones de prueba'); axes[0].legend()

axes[1].plot(x_v, f_test*Lg, COLORS[0], label='$f \\cdot Lg$ (LHS)')
axes[1].plot(x_v, Lf*g_test, COLORS[1], ls='--', label='$(Lf)\\cdot g$ (RHS)')
axes[1].set_xlabel('$x$'); axes[1].set_ylabel('Integrando')
axes[1].set_title(f'<f|Lg> = {lhs_3:.6f},  <Lf|g> = {rhs_3:.6f}')
axes[1].legend()
axes[1].text(1.5, axes[1].get_ylim()[1]*0.7,
             f'Diferencia = {abs(lhs_3-rhs_3):.2e}  ok',
             fontsize=12, bbox=dict(facecolor='lightgreen', alpha=0.8))

plt.tight_layout(); plt.show()
print(f"<f | Lg> = {lhs_3:.8f}")
print(f"<Lf| g> = {rhs_3:.8f}")
print(f"Diferencia = {abs(lhs_3-rhs_3):.2e}  operador hermitco verificado")


---
## Problema 4: Autofunciones y Autovalores de Problemas de Sturm-Liouville

| Caso | Ecuación | Dominio | CCs | $\lambda_n$ | $y_n(x)$ |
|------|----------|---------|-----|-------------|----------|
| (a) | $x^2y''+5xy'+\lambda y=0$ | $[1,e]$ | $y(1)=y(e)=0$ | $4+n^2\pi^2$ | $x^{-2}\sin(n\pi\ln x)$ |
| (b) | $x^2y''+axy'+\lambda y=0$ | $[1,e]$ | $y(1)=y(e)=0$ | $\tfrac{(a-1)^2}{4}+n^2\pi^2$ | $x^{(1-a)/2}\sin(n\pi\ln x)$ |
| (c) | $xy''+y'+\tfrac{\lambda}{x}y=0$ | $[1,2]$ | $y(1)=y(2)=0$ | $(n\pi/\ln 2)^2$ | $\sin(n\pi\ln x/\ln 2)$ |
| (d) | $(3+x)^2y''+2(3+x)y'+\lambda y=0$ | $[-2,1]$ | $y(-2)=y(1)=0$ | $\tfrac{1}{4}[1+(n\pi/\ln2)^2]$ | $(3+x)^{-1/2}\sin(n\pi\ln(3+x)/(2\ln2))$ |
| (e) | $y''+\lambda^2 y=0$ | $[0,L]$ | $y'(0)=y(L)=0$ | $(2n+1)\pi/(2L)$ | $\cos\!\frac{(2n+1)\pi x}{2L}$ |
| (f) | $y''+\lambda^2 y=0$ | $[0,L]$ | $y(0)=y(L)=0$ | $n\pi/L$ | $\sin\!\frac{n\pi x}{L}$ |
| (g) | $y''+\lambda^2 y=0$ | $[0,L]$ | $y'(0)=y'(L)=0$ | $n\pi/L$ | $\cos\!\frac{n\pi x}{L}$ |


In [ ]:
# P4 Casos (a) y (b): Euler-Cauchy en [1, e]
x_ab = np.linspace(1, np.e, 500)
ln2  = np.log(2)
N_modes = 5

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Problema 4 — Casos (a) y (b)\nEcuacion de Euler-Cauchy en $[1, e]$',
             fontsize=15, fontweight='bold')

ax = axes[0,0]
for n in range(1, N_modes+1):
    lam = 4 + (n*np.pi)**2
    yn  = x_ab**(-2) * np.sin(n*np.pi*np.log(x_ab))
    ax.plot(x_ab, yn, label=f'$n={n}$, $\\lambda={lam:.1f}$')
ax.set_title(r'Caso (a): $y_n=x^{-2}\sin(n\pi\ln x)$')
ax.set_xlabel('$x$'); ax.set_ylabel('$y_n(x)$')
ax.axhline(0, color='k', lw=0.8); ax.legend(fontsize=8)

ax = axes[0,1]
a_vals = [1,2,3,5]
for a_val, color in zip(a_vals, COLORS):
    lam1 = (a_val-1)**2/4 + np.pi**2
    yn1  = x_ab**((1-a_val)/2) * np.sin(np.pi*np.log(x_ab))
    ax.plot(x_ab, yn1, color=color, label=f'$a={a_val}$, $\\lambda_1={lam1:.1f}$')
ax.set_title(r'Caso (b): $n=1$, distintos $a$')
ax.set_xlabel('$x$'); ax.set_ylabel('$y_1(x)$')
ax.axhline(0, color='k', lw=0.8); ax.legend(fontsize=9)

ns_arr = np.arange(1, 11)
ax = axes[1,0]
for a_val, color in zip(a_vals, COLORS):
    lams = (a_val-1)**2/4 + (ns_arr*np.pi)**2
    ax.plot(ns_arr, lams, 'o-', color=color, label=f'$a={a_val}$')
ax.set_title(r'Autovalores $\lambda_n$ vs. $n$')
ax.set_xlabel('$n$'); ax.set_ylabel(r'$\lambda_n$'); ax.legend()

ax = axes[1,1]
pairs = [(1,2),(1,3),(2,3),(1,1),(2,2)]
vals  = []
for n,m in pairs:
    yn = x_ab**(-2)*np.sin(n*np.pi*np.log(x_ab))
    ym = x_ab**(-2)*np.sin(m*np.pi*np.log(x_ab))
    vals.append(np.trapezoid(x_ab**3*yn*ym, x_ab))
labels = [f'$(n,m)=({n},{m})$' for n,m in pairs]
cols   = ['steelblue' if abs(v)<0.05 else 'tomato' for v in vals]
ax.bar(labels, vals, color=cols)
ax.axhline(0, color='k', lw=0.8)
ax.set_title(r'Ortogonalidad: $\int_1^e x^3 y_n y_m dx$ (caso a)')
ax.set_ylabel('Valor integral'); ax.set_xticklabels(labels, rotation=20, fontsize=9)

plt.tight_layout(); plt.show()


In [ ]:
# P4 Casos (c) y (d)
x_c = np.linspace(1, 2, 500)
x_d = np.linspace(-2, 1, 500)
ln2 = np.log(2)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Problema 4 — Casos (c) y (d)', fontsize=15, fontweight='bold')

ax = axes[0,0]
for n in range(1,6):
    lam_c = (n*np.pi/ln2)**2
    yn = np.sin(n*np.pi*np.log(x_c)/ln2)
    ax.plot(x_c, yn, label=f'$n={n}$, $\\lambda={lam_c:.1f}$')
ax.set_title(r'Caso (c): $y_n=\sin(n\pi\ln x/\ln 2)$, $x\in[1,2]$')
ax.set_xlabel('$x$'); ax.set_ylabel('$y_n(x)$')
ax.axhline(0, color='k', lw=0.8); ax.legend(fontsize=8)

ax = axes[0,1]
pairs_c = [(1,2),(1,3),(2,3),(1,1),(2,2)]
vals_c  = []
for n,m in pairs_c:
    yn = np.sin(n*np.pi*np.log(x_c)/ln2)
    ym = np.sin(m*np.pi*np.log(x_c)/ln2)
    vals_c.append(np.trapezoid(yn*ym/x_c, x_c))
labels_c = [f'$({n},{m})$' for n,m in pairs_c]
cols_c   = ['steelblue' if abs(v)<0.02 else 'tomato' for v in vals_c]
ax.bar(labels_c, vals_c, color=cols_c)
ax.axhline(0, color='k', lw=0.8)
ax.set_title(r'Caso (c): $\int_1^2\frac{y_n y_m}{x}dx$ (peso $1/x$)')
ax.set_ylabel('Integral')

ax = axes[1,0]
for n in range(1,6):
    lam_d = 0.25*(1+(n*np.pi/ln2)**2)
    yn_d  = (3+x_d)**(-0.5)*np.sin(n*np.pi*np.log(3+x_d)/(2*ln2))
    ax.plot(x_d, yn_d, label=f'$n={n}$, $\\lambda\\approx{lam_d:.1f}$')
ax.set_title(r'Caso (d): $y_n=\frac{\sin(n\pi\ln(3+x)/(2\ln2))}{\sqrt{3+x}}$, $x\in[-2,1]$')
ax.set_xlabel('$x$'); ax.set_ylabel('$y_n(x)$')
ax.axhline(0, color='k', lw=0.8); ax.legend(fontsize=8)

ax = axes[1,1]
ns_arr = np.arange(1, 8)
lams_a = 4+(ns_arr*np.pi)**2
lams_c = (ns_arr*np.pi/ln2)**2
lams_d = 0.25*(1+(ns_arr*np.pi/ln2)**2)
ax.semilogy(ns_arr, lams_a, 'o-', color=COLORS[0], label='Caso (a)')
ax.semilogy(ns_arr, lams_c, 's-', color=COLORS[1], label='Caso (c)')
ax.semilogy(ns_arr, lams_d, '^-', color=COLORS[2], label='Caso (d)')
ax.set_title('Espectro de autovalores (escala log)')
ax.set_xlabel('$n$'); ax.set_ylabel(r'$\lambda_n$'); ax.legend()

plt.tight_layout(); plt.show()


In [ ]:
# P4 Casos (e), (f), (g): y'' + lam^2 y = 0 en [0, L]
L = np.pi
x_efg = np.linspace(0, L, 500)
N_show = 5

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(fr'Problema 4 — Casos (e), (f), (g): $y\'\' + \lambda^2 y = 0$ en $[0, L=\pi]$',
             fontsize=14, fontweight='bold')

# Caso (e): y'(0)=0, y(L)=0
ax = axes[0,0]
for n in range(N_show):
    lam = (2*n+1)*np.pi/(2*L)
    yn  = np.cos(lam*x_efg)
    ax.plot(x_efg/L, yn, label=f'$n={n}$')
ax.set_title(r"Caso (e): $y'(0)=0,\;y(L)=0$" + '\n' + r"$y_n=\cos\!\frac{(2n+1)\pi x}{2L}$")
ax.set_xlabel('$x/L$'); ax.set_ylabel('$y_n(x)$')
ax.axhline(0, color='k', lw=0.8); ax.legend(fontsize=8)

# Caso (f): y(0)=0, y(L)=0
ax = axes[0,1]
for n in range(1, N_show+1):
    lam = n*np.pi/L
    yn  = np.sin(lam*x_efg)
    ax.plot(x_efg/L, yn, label=f'$n={n}$')
ax.set_title(r'Caso (f): $y(0)=0,\;y(L)=0$' + '\n' + r'$y_n=\sin\!\frac{n\pi x}{L}$')
ax.set_xlabel('$x/L$'); ax.set_ylabel('$y_n(x)$')
ax.axhline(0, color='k', lw=0.8); ax.legend(fontsize=8)

# Caso (g): y'(0)=0, y'(L)=0
ax = axes[0,2]
for n in range(N_show):
    lam = n*np.pi/L
    yn  = np.cos(lam*x_efg)
    ax.plot(x_efg/L, yn, label=f'$n={n}$')
ax.set_title(r"Caso (g): $y'(0)=0,\;y'(L)=0$" + '\n' + r'$y_n=\cos\!\frac{n\pi x}{L}$, $n\geq 0$')
ax.set_xlabel('$x/L$'); ax.set_ylabel('$y_n(x)$')
ax.axhline(0, color='k', lw=0.8); ax.legend(fontsize=8)

# Ortogonalidad caso (f)
ax = axes[1,0]
N_ort = 5
Omat = np.zeros((N_ort, N_ort))
for i in range(1, N_ort+1):
    for j in range(1, N_ort+1):
        yi = np.sin(i*np.pi*x_efg/L); yj = np.sin(j*np.pi*x_efg/L)
        Omat[i-1,j-1] = np.trapezoid(yi*yj, x_efg)
im = ax.imshow(Omat, cmap='RdBu_r', aspect='auto', vmin=-0.1, vmax=np.pi/2+0.1)
plt.colorbar(im, ax=ax, label=r'$\int_0^L y_n y_m dx$')
ax.set_title(r'Ortogonalidad (f): $\int_0^L\sin\frac{n\pi x}{L}\sin\frac{m\pi x}{L}dx$')
ax.set_xticks(range(N_ort)); ax.set_xticklabels([f'$n={i+1}$' for i in range(N_ort)])
ax.set_yticks(range(N_ort)); ax.set_yticklabels([f'$n={i+1}$' for i in range(N_ort)])

# Autovalores de los tres casos
ax = axes[1,1]
ns_efg = np.arange(0, 8)
ax.plot(ns_efg, ((2*ns_efg+1)*np.pi/(2*L))**2, 'o-', color=COLORS[0], label='(e)')
ax.plot(ns_efg, ((ns_efg+1)*np.pi/L)**2,        's-', color=COLORS[1], label='(f)')
ax.plot(ns_efg, (ns_efg*np.pi/L)**2,             '^-', color=COLORS[2], label='(g)')
ax.set_title(r'Espectro $\lambda_n^2$ para casos (e), (f), (g)')
ax.set_xlabel('Indice'); ax.set_ylabel(r'$\lambda_n^2$'); ax.legend()

# Comparacion nodos interiores
ax = axes[1,2]
for n in range(1,5):
    yn_f = np.sin(n*np.pi*x_efg/L)
    yn_g = np.cos(n*np.pi*x_efg/L)
    ax.plot(x_efg/L, yn_f, color=COLORS[n-1], ls='-',  label=f'(f) $n={n}$')
    ax.plot(x_efg/L, yn_g, color=COLORS[n-1], ls='--', alpha=0.5)
ax.axhline(0, color='k', lw=0.8)
ax.set_title('Nodos interiores\n(solido: senos, punteado: cosenos)')
ax.set_xlabel('$x/L$'); ax.set_ylabel('$y_n$'); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()


---
## Problema 5: Ecuación de Euler — No Genera Base Ortogonal

**Enunciado:** La ecuación $\rho^2\ddot{R}+\rho\dot{R}-\nu^2 R=0$ tiene soluciones $R_\nu=\rho^\nu$.
Demostrar que $[qW]_a^b \neq 0$ y que $\int_a^b R_\nu R_{\nu'} d\rho \neq 0$.

### Resultados analíticos

$$q(\rho)=\rho, \quad W(R_\nu,R_{\nu'}) = (\nu'-\nu)\rho^{\nu+\nu'-1}$$

$$[qW]_a^b = (\nu'-\nu)(b^{\nu+\nu'}-a^{\nu+\nu'}) \neq 0$$

$$\int_a^b\rho^{\nu+\nu'}d\rho = \frac{b^{\nu+\nu'+1}-a^{\nu+\nu'+1}}{\nu+\nu'+1} \neq 0$$


In [ ]:
# P5-1: Visualizacion de R_nu y no-ortogonalidad
a_p5, b_p5 = 1.0, 3.0
rho = np.linspace(a_p5, b_p5, 500)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(r'Problema 5: Soluciones $R_\nu(\rho)=\rho^\nu$ — No Ortogonalidad',
             fontsize=15, fontweight='bold')

ax = axes[0,0]
for nu_val, color in zip([0.5,1.0,1.5,2.0,2.5,3.0], COLORS):
    ax.plot(rho, rho**nu_val, color=color, label=f'$\\nu={nu_val}$')
ax.set_title(r'$R_\nu(\rho)=\rho^\nu$ para distintos $\nu$')
ax.set_xlabel(r'$\rho$'); ax.set_ylabel(r'$R_\nu$'); ax.legend()

nu_arr   = np.linspace(0.2, 3.0, 80)
nu_prime = 1.5
qW_diff  = (nu_prime-nu_arr)*(b_p5**(nu_arr+nu_prime) - a_p5**(nu_arr+nu_prime))

ax = axes[0,1]
ax.plot(nu_arr, qW_diff, 'steelblue', label=f"$[qW]_a^b$, $\\nu'={nu_prime}$")
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_title(r"Termino de frontera $[qW]_a^b = (\nu'-\nu)(b^{\nu+\nu'}-a^{\nu+\nu'})$")
ax.set_xlabel(r'$\nu$'); ax.set_ylabel(r'$[qW]_a^b$'); ax.legend()
ax.text(1.5, qW_diff.max()*0.7,
        r'Nunca es cero para $\nu\neq\nu\prime$!',
        fontsize=11, color='red', ha='center')

ax = axes[1,0]
integrals = []
for nu_val in nu_arr:
    exp = nu_val + nu_prime + 1
    I   = (b_p5**exp - a_p5**exp)/exp if abs(exp)>1e-10 else np.log(b_p5/a_p5)
    integrals.append(I)
ax.plot(nu_arr, integrals, 'steelblue', lw=2, label='Analitica')
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.axvline(nu_prime, color='orange', lw=1, ls=':', label=f"$\\nu=\\nu'={nu_prime}$")
ax.set_title(fr"$\int_a^b \rho^{{\nu+\nu'}}d\rho$, $\nu'={nu_prime}$")
ax.set_xlabel(r'$\nu$'); ax.set_ylabel('Integral'); ax.legend()

ax = axes[1,1]
nu_grid = np.linspace(0.2, 3.0, 30)
NU1, NU2 = np.meshgrid(nu_grid, nu_grid)
EXP = NU1+NU2+1
I_map = np.where(np.abs(EXP)>1e-10,
                 (b_p5**EXP-a_p5**EXP)/EXP,
                 np.log(b_p5/a_p5))
im = ax.pcolormesh(nu_grid, nu_grid, I_map, cmap='RdBu_r', shading='auto')
plt.colorbar(im, ax=ax, label=r"$\int_a^b\rho^{\nu+\nu'}d\rho$")
ax.set_title(r"Mapa: $\int_a^b\rho^{\nu+\nu\prime}d\rho$ -- nunca es cero")
ax.set_xlabel(r'$\nu$'); ax.set_ylabel(r"$\nu'$")
ax.plot(nu_grid, nu_grid, 'w--', lw=1.5, label=r"$\nu=\nu'$"); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()


In [ ]:
# P5-2: Contraste con Bessel (SI ortogonal)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Problema 5: Contraste — Bessel (ortogonal) vs. Potencias (no ortogonal)',
             fontsize=14, fontweight='bold')

xi = jn_zeros(0, 5)
x_01 = np.linspace(0, 1, 500)

ax = axes[0]
for k, z_k in enumerate(xi[:4], start=1):
    y_k  = jv(0, z_k*x_01)
    norm = np.sqrt(np.trapezoid(x_01*y_k**2, x_01))
    ax.plot(x_01, y_k/norm, label=f'$J_0(\\xi_{k}x)$, $\\xi_{k}={z_k:.2f}$')
ax.axhline(0, color='k', lw=0.8)
ax.set_title(r'Funciones de Bessel $J_0(\xi_k x)$ — SI ortogonales con peso $x$')
ax.set_xlabel(r'$x\in[0,1]$'); ax.set_ylabel('Autofuncion normalizada'); ax.legend(fontsize=9)

ax = axes[1]
N_bess = 5
x_01b = np.linspace(0, 1, 2000)
Obess = np.zeros((N_bess, N_bess))
for i, zi in enumerate(xi[:N_bess]):
    for j, zj in enumerate(xi[:N_bess]):
        yi = jv(0, zi*x_01b); yj = jv(0, zj*x_01b)
        Obess[i,j] = np.trapezoid(x_01b*yi*yj, x_01b)
im = ax.imshow(Obess, cmap='RdBu_r', aspect='auto', vmin=-0.05, vmax=0.55)
plt.colorbar(im, ax=ax, label=r'$\int_0^1 x J_0(\xi_i x)J_0(\xi_j x)dx$')
ax.set_title(r'Ortogonalidad de Bessel con peso $p(x)=x$')
ax.set_xticks(range(N_bess)); ax.set_xticklabels([fr'$\xi_{i+1}$' for i in range(N_bess)])
ax.set_yticks(range(N_bess)); ax.set_yticklabels([fr'$\xi_{i+1}$' for i in range(N_bess)])

plt.tight_layout(); plt.show()
print("Diagonal (normas^2):", np.diag(Obess).round(5))
print("Fuera de diagonal (aprox 0):", Obess[0,1:4].round(6))


---
## Problema 6: Funciones Especiales en Forma Autoadjunta

Se convierte cada ecuación a la forma $\frac{d}{dx}[q(x)y'] + \lambda p(x)y = 0$ usando el factor integrante $p = \frac{1}{a_2}\exp\!\!\int\!\frac{a_1}{a_2}dx$.

| Función | $q(x)$ | $p(x)$ |
|---------|--------|--------|
| Bessel | $x$ | $x$ |
| Legendre | $1-x^2$ | $1$ |
| Hermite | $e^{-x^2}$ | $e^{-x^2}$ |
| Laguerre | $xe^{-x}$ | $e^{-x}$ |
| Lag. Asoc. | $x^{k+1}e^{-x}$ | $x^k e^{-x}$ |
| Chebyshev T | $(1-x^2)^{1/2}$ | $(1-x^2)^{-1/2}$ |
| Chebyshev U | $(1-x^2)^{3/2}$ | $(1-x^2)^{1/2}$ |
| Gegenbauer | $(1-x^2)^{\beta+1}$ | $(1-x^2)^\beta$ |


In [ ]:
# P6-A: Ecuacion de Bessel
x_bs = np.linspace(0.01, 20, 2000)
x_01 = np.linspace(0, 1, 3000)
xi5  = jn_zeros(0, 5)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Problema 6(a): Ecuacion de Bessel\n'
             r'$\frac{d}{dx}[x\frac{dy}{dx}] + (k^2x - n^2/x)y = 0$',
             fontsize=14, fontweight='bold')

ax = axes[0]
for n_val in range(5):
    ax.plot(x_bs, jv(n_val, x_bs), label=f'$J_{n_val}(x)$')
ax.axhline(0, color='k', lw=0.7)
ax.set_xlim(0,20); ax.set_ylim(-0.5,1.1)
ax.set_title('Funciones de Bessel $J_n(x)$')
ax.set_xlabel('$x$'); ax.set_ylabel('$J_n(x)$'); ax.legend()

ax = axes[1]
x_w = np.linspace(0,10,500)
ax.plot(x_w, x_w, 'steelblue', lw=2.5, label='$p(x)=x$')
ax.fill_between(x_w, x_w, alpha=0.15, color='steelblue')
ax.set_title('Factor de peso $p(x) = x$')
ax.set_xlabel('$x$'); ax.set_ylabel('$p(x)$'); ax.legend()

ax = axes[2]
N5 = 5
Omat = np.zeros((N5,N5))
for i in range(N5):
    for j in range(N5):
        yi = jv(0,xi5[i]*x_01); yj = jv(0,xi5[j]*x_01)
        Omat[i,j] = np.trapezoid(x_01*yi*yj, x_01)
im = ax.imshow(Omat, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax)
ax.set_title(r'$\int_0^1 x J_0(\xi_i x)J_0(\xi_j x)dx$')
ax.set_xticks(range(N5)); ax.set_xticklabels([fr'$\xi_{i+1}$' for i in range(N5)], fontsize=9)
ax.set_yticks(range(N5)); ax.set_yticklabels([fr'$\xi_{i+1}$' for i in range(N5)], fontsize=9)

plt.tight_layout(); plt.show()


In [ ]:
# P6-B/C: Legendre y Legendre Asociada
from numpy.polynomial.legendre import legval

x_leg = np.linspace(-1, 1, 500)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Problema 6(b)/(c): Ecuaciones de Legendre y Legendre Asociada\n'
             r'$\frac{d}{dx}[(1-x^2)\frac{dy}{dx}] + l(l+1)y = 0$',
             fontsize=13, fontweight='bold')

ax = axes[0,0]
for l_val in range(6):
    coeffs = np.zeros(l_val+1); coeffs[-1] = 1
    Pl = legval(x_leg, coeffs)
    ax.plot(x_leg, Pl, label=f'$P_{l_val}(x)$')
ax.axhline(0, color='k', lw=0.7)
ax.set_title('Polinomios de Legendre $P_l(x)$')
ax.set_xlabel('$x$'); ax.set_ylabel('$P_l(x)$'); ax.set_ylim(-1.1,1.1); ax.legend(fontsize=9)

ax = axes[0,1]
ax.plot(x_leg, 1-x_leg**2, 'steelblue', lw=2.5, label=r'$q(x)=1-x^2$')
ax.fill_between(x_leg, 1-x_leg**2, alpha=0.15, color='steelblue')
ax.axhline(0, color='k', lw=0.7)
ax.set_title(r'Factor $q(x)=1-x^2$ (se anula en $x=\pm1$)')
ax.set_xlabel('$x$'); ax.set_ylabel('$q(x)$'); ax.legend()

ax = axes[1,0]
for (l_val,m_val), color in zip([(1,1),(2,1),(2,2),(3,1),(3,2),(3,3)], COLORS):
    Plm  = lpmv(m_val, l_val, x_leg)
    norm = np.max(np.abs(Plm)+1e-10)
    ax.plot(x_leg, Plm/norm, color=color, label=f'$P_{l_val}^{{{m_val}}}$ (norm.)')
ax.axhline(0, color='k', lw=0.7)
ax.set_title('Funciones de Legendre Asociadas $P_l^m(x)$ (normalizadas)')
ax.set_xlabel('$x$'); ax.set_ylabel(r'$P_l^m/\max$'); ax.legend(fontsize=8)

ax = axes[1,1]
N_leg = 6
Oleg = np.zeros((N_leg,N_leg))
for i in range(N_leg):
    ci = np.zeros(i+1); ci[-1]=1; Pi = legval(x_leg, ci)
    for j in range(N_leg):
        cj = np.zeros(j+1); cj[-1]=1; Pj = legval(x_leg, cj)
        Oleg[i,j] = np.trapezoid(Pi*Pj, x_leg)
im = ax.imshow(Oleg, cmap='RdBu_r', aspect='auto', vmin=-0.5, vmax=0.8)
plt.colorbar(im, ax=ax, label=r'$\int_{-1}^{1}P_i P_j dx$')
ax.set_title(r'Ortogonalidad: $\int_{-1}^{1}P_i P_j dx = \frac{2}{2l+1}\delta_{ij}$')
lbls = [f'$P_{i}$' for i in range(N_leg)]
ax.set_xticks(range(N_leg)); ax.set_xticklabels(lbls, fontsize=9)
ax.set_yticks(range(N_leg)); ax.set_yticklabels(lbls, fontsize=9)

plt.tight_layout(); plt.show()


In [ ]:
# P6-D: Hermite
x_h = np.linspace(-4, 4, 600)
import math

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Problema 6(d): Ecuacion de Hermite\n'
             r'$\frac{d}{dx}[e^{-x^2}\frac{dy}{dx}] + 2n\,e^{-x^2}\,y = 0$',
             fontsize=13, fontweight='bold')

ax = axes[0]
for n_val in range(6):
    Hn   = eval_hermite(n_val, x_h)
    norm = np.sqrt(np.trapezoid(np.exp(-x_h**2)*Hn**2, x_h))
    ax.plot(x_h, Hn/norm, label=f'$H_{n_val}(x)$')
ax.axhline(0, color='k', lw=0.7)
ax.set_title('Polinomios de Hermite $H_n(x)$ (normalizados)')
ax.set_xlabel('$x$'); ax.set_ylabel('Amplitud normalizada')
ax.set_ylim(-1.5,1.5); ax.legend(fontsize=9)

ax = axes[1]
ax.plot(x_h, np.exp(-x_h**2), 'tomato', lw=2.5, label=r'$p(x)=e^{-x^2}$')
ax.fill_between(x_h, np.exp(-x_h**2), alpha=0.2, color='tomato')
ax.set_title(r'Factor de peso $p(x)=e^{-x^2}$ (Gaussiana)')
ax.set_xlabel('$x$'); ax.set_ylabel('$p(x)$'); ax.legend()

ax = axes[2]
for n_val in range(5):
    Hn  = eval_hermite(n_val, x_h)
    fac = math.factorial(n_val)
    psi_n = (2**n_val * fac * np.sqrt(np.pi))**(-0.5) * np.exp(-x_h**2/2) * Hn
    ax.plot(x_h, psi_n + n_val*0.6, label=f'$n={n_val}$')
ax.set_title('Funciones de onda OAC\n(desplazadas en energia)')
ax.set_xlabel('$x$'); ax.legend(fontsize=9); ax.axhline(0, color='k', lw=0.5)

plt.tight_layout(); plt.show()


In [ ]:
# P6-E/F: Laguerre y Laguerre Asociada
x_lag = np.linspace(0, 15, 600)
x_H   = np.linspace(0, 30, 1000)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Problema 6(e)/(f): Ecuaciones de Laguerre\n'
             r'(e): $\frac{d}{dx}[xe^{-x}\frac{dy}{dx}] + ne^{-x}y = 0$',
             fontsize=12, fontweight='bold')

ax = axes[0,0]
for n_val in range(6):
    Ln = eval_laguerre(n_val, x_lag)
    ax.plot(x_lag, Ln, label=f'$L_{n_val}(x)$')
ax.axhline(0, color='k', lw=0.7)
ax.set_xlim(0,12); ax.set_ylim(-6,6)
ax.set_title('Polinomios de Laguerre $L_n(x)$')
ax.set_xlabel('$x$'); ax.set_ylabel('$L_n(x)$'); ax.legend(fontsize=9)

ax = axes[0,1]
ax.plot(x_lag, np.exp(-x_lag), 'purple', lw=2.5, label=r'$p(x)=e^{-x}$')
ax.plot(x_lag, x_lag*np.exp(-x_lag), 'green', lw=2.5, ls='--', label=r'$q(x)=xe^{-x}$')
ax.fill_between(x_lag, np.exp(-x_lag), alpha=0.15, color='purple')
ax.set_title(r'Factores $p(x)=e^{-x}$ y $q(x)=xe^{-x}$')
ax.set_xlabel('$x$'); ax.legend()

ax = axes[1,0]
for k_val in [0,1,2,3]:
    Lnk  = eval_genlaguerre(2, k_val, x_lag)
    norm = np.sqrt(np.trapezoid(np.exp(-x_lag)*x_lag**k_val*Lnk**2, x_lag)+1e-20)
    ax.plot(x_lag[:300], (Lnk/norm)[:300],
            label=f'$L_2^{{{k_val}}}(x)$ (norm.)')
ax.axhline(0, color='k', lw=0.7)
ax.set_title(r'Laguerre Asociada $L_n^k(x)$, $n=2$, distintos $k$')
ax.set_xlabel('$x$'); ax.legend(fontsize=9)

ax = axes[1,1]
configs = [(1,0,'steelblue'),(2,0,'tomato'),(2,1,'green'),(3,0,'purple'),(3,1,'orange')]
for (n_val, l_val, color) in configs:
    rho_H = 2*x_H/n_val
    Lnl   = eval_genlaguerre(n_val-l_val-1, 2*l_val+1, rho_H)
    R_nl  = rho_H**l_val * np.exp(-rho_H/2) * Lnl
    norm  = np.sqrt(np.trapezoid(R_nl**2*x_H**2, x_H)+1e-20)
    ax.plot(x_H, R_nl/norm, color=color, label=f'$R_{{{n_val}{l_val}}}$')
ax.axhline(0, color='k', lw=0.7)
ax.set_xlim(0,25); ax.set_ylim(-1,1.5)
ax.set_title(r'Parte radial $R_{nl}(r)$ — atomo H')
ax.set_xlabel(r'$r/a_0$'); ax.set_ylabel(r'$R_{nl}$'); ax.legend(fontsize=9)

plt.tight_layout(); plt.show()


In [ ]:
# P6-G/H: Chebyshev 1a y 2a especie
x_chb  = np.linspace(-1, 1, 600)
x_chbs = np.linspace(-0.999, 0.999, 600)
theta_chb = np.arccos(x_chbs)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Problema 6(g)/(h): Ecuaciones de Chebyshev\n'
             r'1a especie: $p=(1-x^2)^{-1/2}$; 2a especie: $p=(1-x^2)^{1/2}$',
             fontsize=13, fontweight='bold')

ax = axes[0,0]
for n_val in range(6):
    Tn = np.cos(n_val*np.arccos(x_chb))
    ax.plot(x_chb, Tn, label=f'$T_{n_val}(x)$')
ax.axhline(0, color='k', lw=0.7)
ax.set_title(r'Chebyshev 1a: $T_n(x)=\cos(n\arccos x)$')
ax.set_xlabel('$x$'); ax.set_ylabel('$T_n(x)$'); ax.set_ylim(-1.1,1.1); ax.legend(fontsize=9)

ax = axes[0,1]
for n_val in range(6):
    with np.errstate(divide='ignore', invalid='ignore'):
        Un = np.where(np.abs(np.sin(theta_chb))>1e-10,
                      np.sin((n_val+1)*theta_chb)/np.sin(theta_chb),
                      (n_val+1)*np.cos((n_val+1)*theta_chb)/np.cos(theta_chb))
    ax.plot(x_chbs, Un, label=f'$U_{n_val}(x)$')
ax.axhline(0, color='k', lw=0.7)
ax.set_title(r'Chebyshev 2a: $U_n(x)=\sin((n+1)\arccos x)/\sin(\arccos x)$')
ax.set_xlabel('$x$'); ax.set_ylabel('$U_n(x)$'); ax.legend(fontsize=9)

ax = axes[1,0]
ax.plot(x_chbs, 1/np.sqrt(1-x_chbs**2), 'steelblue', lw=2.5,
        label=r'$p_T=(1-x^2)^{-1/2}$ (1a)')
ax.plot(x_chbs, np.sqrt(1-x_chbs**2), 'tomato', lw=2.5, ls='--',
        label=r'$p_U=(1-x^2)^{1/2}$ (2a)')
ax.set_ylim(0,5); ax.set_title('Factores de peso de Chebyshev')
ax.set_xlabel('$x$'); ax.set_ylabel('$p(x)$'); ax.legend()

ax = axes[1,1]
N_chb = 5
Ochb = np.zeros((N_chb,N_chb))
for i in range(N_chb):
    Ti = np.cos(i*np.arccos(x_chbs))
    for j in range(N_chb):
        Tj = np.cos(j*np.arccos(x_chbs))
        Ochb[i,j] = np.trapezoid(Ti*Tj/np.sqrt(1-x_chbs**2), x_chbs)
im = ax.imshow(Ochb, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax, label=r'$\int_{-1}^{1}T_i T_j/\sqrt{1-x^2}dx$')
ax.set_title(r'Ortogonalidad $T_i T_j$ con peso $(1-x^2)^{-1/2}$')
lbls = [f'$T_{i}$' for i in range(N_chb)]
ax.set_xticks(range(N_chb)); ax.set_xticklabels(lbls, fontsize=9)
ax.set_yticks(range(N_chb)); ax.set_yticklabels(lbls, fontsize=9)

plt.tight_layout(); plt.show()


In [ ]:
# P6-K: Gegenbauer y resumen de funciones especiales
x_geg = np.linspace(-1, 1, 500)
x_pos_g = np.linspace(0, 8, 600)
x_pos_l = np.linspace(-0.999, 0.999, 600)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Problema 6(k): Gegenbauer + Resumen de Factores de Peso',
             fontsize=14, fontweight='bold')

ax = axes[0,0]
for beta_val, color in zip([0, 0.5, 1, 2], COLORS):
    for n_val, ls in [(2,'-'),(3,'--')]:
        try:
            Cn   = gegenbauer(n_val, beta_val+0.5)(x_geg)
            norm = np.max(np.abs(Cn))+1e-10
            ax.plot(x_geg, Cn/norm, color=color, ls=ls,
                    label=f'$C_{n_val}^{{(\\beta={beta_val})}}$' if ls=='-' else None)
        except Exception:
            pass
ax.axhline(0, color='k', lw=0.7)
ax.set_title(r'Polinomios de Gegenbauer $C_n^{(\beta)}(x)$')
ax.set_xlabel('$x$'); ax.set_ylabel('Normalizado al max'); ax.legend(fontsize=8)

ax = axes[0,1]
for beta_val, color in zip([0,0.5,1,2,3], COLORS):
    ax.plot(x_geg, (1-x_geg**2)**beta_val, color=color, label=f'$\\beta={beta_val}$')
ax.set_title(r'Factor de peso $p(x)=(1-x^2)^\beta$')
ax.set_xlabel('$x$'); ax.set_ylabel('$p(x)$'); ax.legend()

# Pesos en [-1,1]
ax = axes[1,0]
ax.plot(x_pos_l, np.ones_like(x_pos_l),          COLORS[0], lw=2, label='Legendre $p=1$')
ax.plot(x_pos_l, 1/np.sqrt(1-x_pos_l**2),        COLORS[1], lw=2, label=r'Chebyshev T $(1-x^2)^{-1/2}$')
ax.plot(x_pos_l, np.sqrt(1-x_pos_l**2),          COLORS[2], lw=2, label=r'Chebyshev U $(1-x^2)^{1/2}$')
ax.plot(x_pos_l, (1-x_pos_l**2)**1,              COLORS[3], lw=2, ls='--', label=r'Gegenbauer $\beta=1$')
ax.set_ylim(-0.1,4); ax.legend(fontsize=8)
ax.set_title(r'Pesos en $[-1,1]$')
ax.set_xlabel('$x$'); ax.set_ylabel('$p(x)$')

# Pesos en [0, inf)
ax = axes[1,1]
ax.plot(x_pos_g, np.exp(-x_pos_g**2),         COLORS[0], lw=2, label=r'Hermite $e^{-x^2}$')
ax.plot(x_pos_g, np.exp(-x_pos_g),            COLORS[1], lw=2, label=r'Laguerre $e^{-x}$')
ax.plot(x_pos_g, x_pos_g*np.exp(-x_pos_g),   COLORS[2], lw=2, label=r'Lag. Asoc. $xe^{-x}$')
ax.plot(x_pos_g, x_pos_g**2*np.exp(-x_pos_g),COLORS[3], lw=2, ls='--', label=r'Lag. Asoc. $x^2e^{-x}$')
ax.legend(fontsize=8); ax.set_xlim(0,8)
ax.set_title(r'Pesos en $[0,+\infty)$')
ax.set_xlabel('$x$'); ax.set_ylabel('$p(x)$')

plt.tight_layout(); plt.show()


In [ ]:
# Tabla resumen visual: todos los factores de peso
fig, ax = plt.subplots(figsize=(14, 6))
ax.axis('off')
fig.suptitle('Resumen: Funciones Especiales en Forma Autoadjunta\n'
             r'$\frac{d}{dx}[q(x)\frac{dy}{dx}] + \lambda\,p(x)\,y = 0$',
             fontsize=14, fontweight='bold', y=1.02)

tabla_data = [
    ['Funcion',        'Dominio',       'q(x)',            'p(x)',            r'$\lambda$'],
    ['Bessel',         r'$[0,\infty)$', '$x$',             '$x$',             r'$k^2$'],
    ['Legendre',       '[-1,1]',        r'$1-x^2$',        '1',               r'$l(l+1)$'],
    ['Leg. Asoc.',     '[-1,1]',        r'$1-x^2$',        '1',               r'$l(l+1)$'],
    ['Hermite',        r'$(-\infty,\infty)$', r'$e^{-x^2}$', r'$e^{-x^2}$', r'$2n$'],
    ['Laguerre',       r'$[0,\infty)$', r'$xe^{-x}$',      r'$e^{-x}$',      r'$n$'],
    ['Lag. Asoc.',     r'$[0,\infty)$', r'$x^{k+1}e^{-x}$',r'$x^k e^{-x}$', r'$n$'],
    ['Chebyshev T',    '(-1,1)',         r'$(1-x^2)^{1/2}$',r'$(1-x^2)^{-1/2}$',r'$n^2$'],
    ['Chebyshev U',    '(-1,1)',         r'$(1-x^2)^{3/2}$',r'$(1-x^2)^{1/2}$',r'$n(n+2)$'],
    ['Gegenbauer',     '(-1,1)',         r'$(1-x^2)^{\beta+1}$',r'$(1-x^2)^\beta$',r'$n(n+2\beta+1)$'],
    ['Hipergeom.',     '(0,1)',          r'$x^c(1-x)^{a+b-c+1}$',r'$x^{c-1}(1-x)^{a+b-c}$',r'$-ab$'],
    ['Hipergeom. C.',  r'$[0,\infty)$', r'$x^c e^{-x}$',  r'$x^{c-1}e^{-x}$',r'$-a$'],
]
col_labels = tabla_data[0]
cell_data  = tabla_data[1:]
t = ax.table(cellText=cell_data, colLabels=col_labels,
             loc='center', cellLoc='center')
t.auto_set_font_size(False); t.set_fontsize(10); t.scale(1.2, 1.7)
for j in range(len(col_labels)):
    t[0,j].set_facecolor('#2060a0')
    t[0,j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(cell_data)+1):
    for j in range(len(col_labels)):
        t[i,j].set_facecolor('#f0f6ff' if i%2==0 else 'white')

plt.tight_layout(); plt.show()


---
## Conclusiones

Este notebook presentó visualizaciones para los seis problemas del Taller 5, centrados en la **Teoría de Sturm-Liouville** y su conexión con la mecánica cuántica.

| Problema | Concepto clave | Visualizaciones |
|----------|----------------|----------------|
| **1** | $\langle A\rangle\in\mathbb{R}$ ↔ Hermiticidad | Autovalores reales; oscilación de $\langle\hat{x}\rangle(t)$ |
| **2** | $\hat{p}$, $\hat{L}$ autoadjuntos | Decaimiento en infinito; periodicidad en $\phi$; armónicos esféricos |
| **3** | Factor integrante → forma autoadjunta | Perfiles $q(x)$, $p(x)$; verificación numérica de hermiticidad |
| **4** | Autofunciones y autovalores S-L | 7 problemas: modos, nodos, ortogonalidad |
| **5** | Euler no genera base ortogonal | $[qW]_a^b\neq 0$; integrales no nulas; contraste con Bessel |
| **6** | Funciones especiales en forma autoadjunta | Bessel, Legendre, Hermite, Laguerre, Chebyshev, Gegenbauer |

### Mensaje central

La propiedad de **hermiticidad** (términos de frontera nulos) garantiza que las autofunciones de un operador de Sturm-Liouville forman una **base ortogonal completa** con respecto al factor de peso $p(x)$. Este es el fundamento matemático de la mecánica cuántica y de la expansión en series de funciones especiales.

---
*Física Matemática I — Universidad de Antioquia — 2026*
